# 04 — Representation Search

This notebook demonstrates systematic search over transformation sequences.

**Key idea:** The order and combination of transpiler passes significantly affects the final circuit. WestQuant searches this space systematically rather than relying on fixed optimization levels.

Formally, we search for the optimal compilation policy:

```
pi* = argmin J(C, pi, H)

where:
  pi = (r_0, T_1, r_1, T_2, ..., T_k, r_k)
  J = alpha*D + beta*G_2q + gamma*E + delta*T_c
```

In [ ]:
# Install if needed
# !pip install westquant[qiskit] pandas matplotlib

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from westquant_qiskit import circuit_metrics
from westquant import generate_training_data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("WestQuant Open — Representation Search")
print("=" * 50)

## Step 1: Baseline — Fixed Optimization Levels

Qiskit provides 4 fixed optimization levels (0-3). These are the baseline against which WestQuant's search competes.

In [ ]:
# Build a QFT-8 circuit
n = 8
qc = QuantumCircuit(n)
qc.append(QFTGate(n), range(n))
qc = qc.decompose(reps=3)

# Baseline: Qiskit's 4 fixed optimization levels
baseline_results = []
for opt_level in range(4):
    compiled = transpile(qc, optimization_level=opt_level, seed_transpiler=42)
    m = circuit_metrics(compiled)
    baseline_results.append({
        "method": f"Qiskit opt={opt_level}",
        "depth": m["depth"],
        "two_qubit_gates": m["two_qubit_gates"],
        "size": m["size"],
        "swap_gates": m["swap_gates"],
    })
    print(f"  Qiskit opt={opt_level}: depth={m['depth']}, 2q={m['two_qubit_gates']}, size={m['size']}")

baseline_df = pd.DataFrame(baseline_results)
print(f"\nBaseline: 4 fixed optimization levels")

## Step 2: WestQuant Search — Varying Compilation Configurations

WestQuant's search varies over optimization levels, basis gates, layout methods, and routing methods — producing many more candidates than the 4 fixed levels.

In [ ]:
# WestQuant search: vary over all compilation dimensions
samples = generate_training_data(
    qc,
    framework="qiskit",
    samples=200,
    seed=42,
    optimization_levels=[0, 1, 2, 3],
    basis_gates_options=[
        ["cx", "u3", "u1", "u2"],
        ["cx", "rz", "sx", "x"],
        ["ecr", "rz", "sx", "x"],
    ],
    layout_methods=["trivial", "dense", "sabre"],
    routing_methods=["sabre", "stochastic", "basic"],
)

search_results = []
for s in samples:
    search_results.append({
        "method": s.representation,
        "depth": s.depth,
        "two_qubit_gates": s.two_qubit_gates,
        "size": s.size,
        "swap_gates": s.swap_gates,
    })

search_df = pd.DataFrame(search_results)
print(f"WestQuant search: {len(search_df)} candidates evaluated")
print(f"Best 2Q gates:  {search_df['two_qubit_gates'].min()}")
print(f"Best depth:     {search_df['depth'].min()}")
print(f"Baseline best 2Q: {baseline_df['two_qubit_gates'].min()}")
print(f"Baseline best depth: {baseline_df['depth'].min()}")

## Step 3: Compare Baseline vs WestQuant Search

In [ ]:
# Find the best from each approach
baseline_best = baseline_df.loc[baseline_df["two_qubit_gates"].idxmin()]
wq_best = search_df.loc[search_df["two_qubit_gates"].idxmin()]

print("=== Fixed vs Searched — QFT-8 ===")
print(f"\nBaseline (Qiskit opt={baseline_best.name if 'name' in baseline_best else 'best'}):")
print(f"  Method:    {baseline_best['method']}")
print(f"  Depth:     {baseline_best['depth']}")
print(f"  2Q gates:  {baseline_best['two_qubit_gates']}")
print(f"  Size:      {baseline_best['size']}")

print(f"\nWestQuant search best:")
print(f"  Method:    {wq_best['method']}")
print(f"  Depth:     {wq_best['depth']}")
print(f"  2Q gates:  {wq_best['two_qubit_gates']}")
print(f"  Size:      {wq_best['size']}")

# Compute improvement
depth_improvement = (baseline_best["depth"] - wq_best["depth"]) / baseline_best["depth"] * 100
gate_improvement = (baseline_best["two_qubit_gates"] - wq_best["two_qubit_gates"]) / baseline_best["two_qubit_gates"] * 100
print(f"\nImprovement:")
print(f"  Depth:     {depth_improvement:+.1f}%")
print(f"  2Q gates:  {gate_improvement:+.1f}%")

In [ ]:
# Pareto front: depth vs 2Q gates
fig, ax = plt.subplots(figsize=(10, 7))

# Plot baseline
ax.scatter(baseline_df["depth"], baseline_df["two_qubit_gates"],
           s=150, color="red", label="Qiskit fixed levels", zorder=5, edgecolors="black")
for _, row in baseline_df.iterrows():
    ax.annotate(row["method"], (row["depth"], row["two_qubit_gates"]),
                textcoords="offset points", xytext=(10, 5), fontsize=9)

# Plot WestQuant search
ax.scatter(search_df["depth"], search_df["two_qubit_gates"],
           s=30, alpha=0.4, color="steelblue", label="WestQuant search")

# Highlight best
ax.scatter(wq_best["depth"], wq_best["two_qubit_gates"],
           s=200, color="green", marker="*", zorder=10, edgecolors="black", label="WestQuant best")
ax.annotate("WQ best", (wq_best["depth"], wq_best["two_qubit_gates"]),
            textcoords="offset points", xytext=(10, 5), fontsize=10, fontweight="bold", color="green")

ax.set_xlabel("Circuit Depth", fontsize=12)
ax.set_ylabel("2-Qubit Gates", fontsize=12)
ax.set_title(f"Representation Search: QFT-{n} — Baseline vs WestQuant", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("representation_search.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4: Multi-Objective Pareto Optimization

The search can target different objectives: depth, 2Q gates, or compile time. The Pareto front shows the trade-off.

In [ ]:
# Compute Pareto front (minimize both depth and 2Q gates)
def pareto_front(df, objectives):
    """Find Pareto-optimal points (minimize all objectives)."""
    is_pareto = [True] * len(df)
    for i in range(len(df)):
        for j in range(len(df)):
            if i == j:
                continue
            # j dominates i if j is better or equal in all objectives and strictly better in at least one
            if all(df.iloc[j][obj] <= df.iloc[i][obj] for obj in objectives) and \
               any(df.iloc[j][obj] < df.iloc[i][obj] for obj in objectives):
                is_pareto[i] = False
                break
    return df[is_pareto]

pareto = pareto_front(search_df, ["depth", "two_qubit_gates"])
print(f"Pareto front: {len(pareto)} optimal points out of {len(search_df)} candidates")
print(pareto.sort_values("depth").to_string(index=False))

## Summary

| Approach | Candidates | Best 2Q | Best Depth |
|----------|-----------|---------|-----------|
| Qiskit fixed levels | 4 | baseline | baseline |
| WestQuant search | 200+ | searched | searched |

The search space is much larger than the 4 fixed optimization levels. WestQuant systematically explores it to find better representations.

**This is Experiment #6 from the project plan:** Fixed vs searched transformation order.